In [3]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from adjustText import adjust_text

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data_loader import load_games_df
from simulation import get_elo_series_prob, simulate_monte_carlo_series

SEASON = "2024-25"
CACHE_PATH = ROOT / "data" / "raw" / f"league_gamelog_{SEASON.replace('-', '_')}.csv"


In [4]:
# Same as notebook: LeagueGameLog -> games_df. Can swap to direct call:
# gamelog = leaguegamelog.LeagueGameLog(season='2024-25', season_type_all_star='Regular Season')
# games_df = gamelog.get_data_frames()[0]

games_df = load_games_df(SEASON, "Regular Season", cache_path=CACHE_PATH)
games_df


ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=120)

In [ ]:
# 1. Split the data
home_games = games_df[games_df['MATCHUP'].str.contains('vs.')].copy()
away_games = games_df[games_df['MATCHUP'].str.contains('@')].copy()


all_games = pd.merge(
    home_games[['GAME_ID', 'GAME_DATE', 'TEAM_ABBREVIATION', 'PTS', 'WL']],
    away_games[['GAME_ID', 'TEAM_ABBREVIATION', 'PTS']],
    on='GAME_ID',
    suffixes=('_HOME', '_AWAY')
)

all_games = all_games.rename(columns={
    'TEAM_ABBREVIATION_HOME': 'HOME_TEAM',
    'TEAM_ABBREVIATION_AWAY': 'AWAY_TEAM',
    'PTS_HOME': 'HOME_PTS',
    'PTS_AWAY': 'AWAY_PTS',
    'WL': 'WL_HOME'
})

print(all_games.head())


In [ ]:
current_elo = {}
for team in all_games.get('HOME_TEAM'):
    if team in current_elo:
        continue
    else:
        current_elo[team]=1500
    if len(current_elo) ==30:
        break

current_elo
ELO_MATURE = current_elo.copy()


In [ ]:
all_games = all_games.sort_values('GAME_DATE')
k_factor=20
elo_history=[]


In [ ]:
for index, row in all_games.iterrows():
    home_team = row['HOME_TEAM']
    away_team = row['AWAY_TEAM']

    home_elo = ELO_MATURE[home_team]
    away_elo = ELO_MATURE[away_team]
    
    elo_diff = home_elo - away_elo 
    
    expected_home = 1 / (10**(-elo_diff / 400) + 1)

    S = 1 if row['WL_HOME'] == 'W' else 0

    # Multiplier 
    margin = abs(row['HOME_PTS'] - row['AWAY_PTS'])
    elo_winner_diff = elo_diff if S == 1 else -elo_diff
    
    mov_mult = ((margin + 3)**0.8) / (7.5 + 0.006 * elo_winner_diff)

    shift = k_factor * mov_mult * (S - expected_home)

    ELO_MATURE[home_team] += shift
    ELO_MATURE[away_team] -= shift


In [ ]:
Static_DNA = (
    games_df.groupby('TEAM_NAME')
    .agg(
        PPG=('PTS', 'mean'),
        STD_PTS=('PTS', 'std'),
        AVG_3PM=('FG3M', 'mean'),
        STD_3PM=('FG3M', 'std'),
        AVG_3PA=('FG3A', 'mean')
    )
    .reset_index()
)


In [ ]:
mapping_df = games_df[['TEAM_NAME', 'TEAM_ABBREVIATION']].drop_duplicates()
name_to_abb = dict(zip(mapping_df['TEAM_NAME'], mapping_df['TEAM_ABBREVIATION']))


In [ ]:
ELO_DNA = Static_DNA.copy()
ELO_DNA['Abb'] = ELO_DNA['TEAM_NAME'].map(name_to_abb)
ELO_DNA['ELO'] = ELO_DNA['Abb'].map(ELO_MATURE)


In [ ]:
plt.figure(figsize=(12, 8))
catter = sns.scatterplot(data = ELO_DNA, x = 'PPG', y='STD_PTS', size = 'AVG_3PM',
                         hue = 'ELO', sizes = (30,500), palette='coolwarm', alpha = 0.7)
texts = []
for i in range(ELO_DNA.shape[0]):
    t = plt.text(
        x=ELO_DNA['PPG'].iloc[i] + 0.05, 
        y=ELO_DNA['STD_PTS'].iloc[i], 
        s=ELO_DNA['Abb'].iloc[i], 
        fontsize=9,
        fontweight='bold'
    )
    texts.append(t)
adjust_text(texts, arrowprops=dict(arrowstyle='->', color='black', lw=0.5))

ppg_median = 114.0
std_median = 12.709365943361048
plt.axvline(x=ppg_median, color='gray', linestyle='--', alpha=0.6, label=f'PPG Median ({ppg_median})')
plt.axhline(y=std_median, color='gray', linestyle='--', alpha=0.6, label=f'STD Median ({std_median:.2f})')

plt.text(ppg_median + 0.5, plt.gca().get_ylim()[1] - 0.5, 'High PPG / High Var', 
         fontsize=10, fontweight='bold', alpha=0.5, verticalalignment='top')

plt.text(ppg_median - 0.5, plt.gca().get_ylim()[1] - 0.5, 'Low PPG / High Var', 
         fontsize=10, fontweight='bold', alpha=0.5, verticalalignment='top', horizontalalignment='right')

plt.text(ppg_median + 0.5, plt.gca().get_ylim()[0] + 0.5, 'High PPG / Low Var', 
         fontsize=10, fontweight='bold', alpha=0.5, verticalalignment='bottom')

plt.text(ppg_median - 0.5, plt.gca().get_ylim()[0] + 0.5, 'Low PPG / Low Var', 
         fontsize=10, fontweight='bold', alpha=0.5, verticalalignment='bottom', horizontalalignment='right')

plt.title("Team Scoring & Volatility Profiles")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
median_ppg = np.median(games_df['PTS'])
median_std = np.std(games_df['PTS'])
median_ppg, median_std


In [ ]:
elo_67 = ELO_DNA['ELO'].quantile(0.67)
elo_33 = ELO_DNA['ELO'].quantile(0.33)

three_std_67 = ELO_DNA['STD_3PM'].quantile(0.67)
three_std_33 = ELO_DNA['STD_3PM'].quantile(0.33)

df_benchmark = ELO_DNA[ELO_DNA['ELO'] >= elo_67]
df_wildcard = ELO_DNA[(ELO_DNA['ELO'] <= elo_33) &(ELO_DNA['STD_3PM'] >= three_std_67)]
df_control = ELO_DNA[(ELO_DNA['ELO'] <= elo_67) &(ELO_DNA['STD_3PM'] <= three_std_33)]


In [ ]:
df_control = df_control.sort_values('ELO', ascending=False)
df_control


In [ ]:
df_benchmark=df_benchmark.sort_values('ELO', ascending=False)
df_benchmark


In [ ]:
df_wildcard = df_wildcard.sort_values('ELO', ascending=False)


In [ ]:
model_1_data = []
for i, giant in df_benchmark.iterrows():
    for j, underdog in df_wildcard.iterrows():
        prob_underdog = 1-(get_elo_series_prob(giant['ELO'], underdog['ELO']))

        model_1_data.append({
            'Team A': giant['TEAM_NAME'],
            'Team B': underdog['TEAM_NAME'],
            'Group': 'Wildcard',
            'Elo_A': giant['ELO'],
            'Elo_B': underdog['ELO'],
            'Elo_Underdog_Prob': prob_underdog
        })
for i, giant in df_benchmark.iterrows():
    for j, underdog in df_control.iterrows():
        prob_underdog = 1-(get_elo_series_prob(giant['ELO'], underdog['ELO']))

        model_1_data.append({
            'Team A': giant['TEAM_NAME'],
            'Team B': underdog['TEAM_NAME'],
            'Group': 'Control',
            'Elo_A': giant['ELO'],
            'Elo_B': underdog['ELO'],
            'Elo_Underdog_Prob': prob_underdog
        })


In [ ]:
elo_results = pd.DataFrame(model_1_data)
elo_results.shape


In [ ]:
master_data = []
# Pairings for the experiment
groups = [(df_wildcard, 'Wildcard'), (df_control, 'Control')]

for df_underdogs, group_name in groups:
    for _, giant in df_benchmark.iterrows():
        for _, underdog in df_underdogs.iterrows():

            # --- MODEL 1: ELO (The Control) ---
            # Requires: ELO_MATURE
            elo_prob = 1-(get_elo_series_prob(giant['ELO'], underdog['ELO']))

            # --- MODEL 2: MONTE CARLO (The Chaos) ---
            # Requires: Full stats for both teams
            sim_prob = (simulate_monte_carlo_series(giant, underdog))

            # --- DATA STORAGE ---
            master_data.append({
                'Matchup': f"{giant['TEAM_NAME']} vs {underdog['TEAM_NAME']}",
                'Group': group_name,
                'Elo_Upset_Prob': elo_prob,
                'Sim_Upset_Prob': sim_prob,
                'Delta': sim_prob - elo_prob # Our Phase 3 key metric
            })

# Convert to the final Research DataFrame
final_results = pd.DataFrame(master_data)
final_results


In [ ]:
control_results = final_results[final_results['Group']=='Control']
wildcard_results = final_results[final_results['Group']=='Wildcard']


In [ ]:
control_results.Sim_Upset_Prob.mean()


In [ ]:
wildcard_results.Sim_Upset_Prob.mean()


In [ ]:
# In-memory replacement for reading df_final_results from Excel
df_final_results = final_results.copy()


In [ ]:
df_final_results["Above_Line"] = df_final_results["Sim_Upset_Prob"] > df_final_results["Elo_Upset_Prob"]


In [ ]:
above = df_final_results[df_final_results["Above_Line"] == True]
below = df_final_results[df_final_results["Above_Line"] == False]


In [ ]:
plt.figure(figsize=(12,8))

plt.scatter(
    above[above["Group"] == "Wildcard"]["Elo_Upset_Prob"],
    above[above["Group"] == "Wildcard"]["Sim_Upset_Prob"],
    color="blue", marker="o", alpha=0.6, label="Underdog > Elo (Wildcard)"
)
plt.scatter(
    above[above["Group"] == "Control"]["Elo_Upset_Prob"],
    above[above["Group"] == "Control"]["Sim_Upset_Prob"],
    color="blue", marker="x", alpha=0.6, label="Underdog > Elo (Control)"
)


plt.scatter(
    below[below["Group"] == "Wildcard"]["Elo_Upset_Prob"],
    below[below["Group"] == "Wildcard"]["Sim_Upset_Prob"],
    color="red", marker="o", alpha=0.6, label="Underdog < Elo (Wildcard)"
)
plt.scatter(
    below[below["Group"] == "Control"]["Elo_Upset_Prob"],
    below[below["Group"] == "Control"]["Sim_Upset_Prob"],
    color="red", marker="x", alpha=0.6, label="Underdog < Elo (Control)"
)
min_val = min(df_final_results["Elo_Upset_Prob"].min(), df_final_results["Sim_Upset_Prob"].min())
max_val = max(df_final_results["Elo_Upset_Prob"].max(), df_final_results["Sim_Upset_Prob"].max())

# create y = x line
x_vals = np.linspace(min_val, max_val, 100)
plt.plot(x_vals, x_vals, linestyle="--", color="black", label="Equal Performance Line (y=x)")

plt.xlabel("Elo Upset Probability")
plt.ylabel("Simulation Upset Probability")
plt.title("Comparison of Elo Model vs Simulation Model Upset Predictions")

# Force equal scaling so the diagonal actually means equal performance
plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.gca().set_aspect('equal', adjustable='box')

plt.legend()
plt.grid(alpha=0.3)

plt.show()
